<div align="center">
  <img src="../images/trucking_banner.png" width="900" alt="Trucking Ontology Workshop"
       onerror="this.style.display='none'" />
</div>

<h1 align="center">🚛 Trucking Ontology Workshop</h1>
<h3 align="center">Notebook 00 — Environment Setup</h3>

<p align="center">
  <img src="https://img.shields.io/badge/python-3.9%2B-blue?logo=python&logoColor=white" />&nbsp;
  <img src="https://img.shields.io/badge/jupyter-notebook-orange?logo=jupyter&logoColor=white" />&nbsp;
</p>

---

> **Purpose:** Run each cell in order to provision all Fabric resources needed for the workshop.  
> Each step is idempotent — safe to re-run if something fails partway through.

## 📋 Prerequisites

#### 🏗️ Workspace & Access

| | Requirement | Detail |
|---|---|---|
| 🏢 | **Fabric-enabled workspace** | Use this workspace for all resources in the tutorial |
| 👤 | **Workspace role: Contributor or higher** | The user running this notebook must have at least Contributor access; Admin role recommended |
| ⚡ | **F16 capacity (minimum)** | Use at least F16 to avoid throttling and ensure optimal performance |

#### 🔧 Tenant Settings *(enabled by a Fabric Administrator)*

| | Setting | Location |
|---|---|---|
| 1️⃣ | Enable **Ontology item** *(preview)* | Admin Portal → Tenant Settings |
| 2️⃣ | User can create **Graph** *(preview)* | Admin Portal → Tenant Settings |
| 3️⃣ | Users can create and share **Data agent** item types *(preview)* | Admin Portal → Tenant Settings |
| 4️⃣ | Users can use **Copilot** and features powered by Azure OpenAI | Admin Portal → Tenant Settings |
| 5️⃣ | Data sent to Azure OpenAI can be **processed outside** your capacity's region | Admin Portal → Tenant Settings |
| 6️⃣ | Data sent to Azure OpenAI can be **stored outside** your capacity's region | Admin Portal → Tenant Settings |

> ⚠️ Settings 4–6 involve cross-region data processing by Azure OpenAI. Review your organization's data residency and compliance policies before enabling them.

---
## 🏗️ Step 1 — Create Fabric Resources

Provisions the Lakehouse that will hold all workshop data.  
Uses [`sempy.fabric`](https://learn.microsoft.com/en-us/python/api/semantic-link-sempy/sempy.fabric), which is pre-installed in Fabric notebooks.

> **Before running:** replace `your-workspace-id` in the config cell with the GUID of your Fabric workspace.  
> You can find it in the workspace URL: `app.fabric.microsoft.com/groups/<workspace-id>/...`

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────
WORKSPACE_ID   = "your-workspace-id"    # 👈 Replace with your Fabric workspace GUID
LAKEHOUSE_NAME = "lh_trucking"          # Lakehouse to create (or reuse if it exists)
# ────────────────────────────────────────────────────────────────────────────

### 🗄️ Create Lakehouse

Creates `lh_trucking` in the workspace.  
If the Lakehouse already exists, creation is skipped and the existing ID is used.

In [ ]:
import sempy.fabric as fabric

try:
    LAKEHOUSE_ID = fabric.get_lakehouse_id(LAKEHOUSE_NAME, workspace=WORKSPACE_ID)
    print(f"ℹ️  Lakehouse '{LAKEHOUSE_NAME}' already exists — skipping creation.")
    print(f"   LAKEHOUSE_ID = {LAKEHOUSE_ID}")
except Exception:
    client = fabric.FabricRestClient()
    resp = client.post(
        f"v1/workspaces/{WORKSPACE_ID}/lakehouses",
        json={"displayName": LAKEHOUSE_NAME},
    )
    resp.raise_for_status()
    LAKEHOUSE_ID = resp.json()["id"]
    print(f"✅ Lakehouse created: {LAKEHOUSE_NAME}")
    print(f"   LAKEHOUSE_ID = {LAKEHOUSE_ID}")

---
## 📦 Step 2 — Generate Reference Data

Generates 11 synthetic JSONL files and writes them directly into the Lakehouse  
at `Files/reference_data/` using the Lakehouse's local filesystem mount.

**Cell 1** downloads the generator script from GitHub into this notebook's built-in resources folder.  
**Cell 2** runs it and confirms all files were written.

In [ ]:
import urllib.request
from pathlib import Path

SCRIPT_URL     = "https://raw.githubusercontent.com/robkerr/trucking-ontology/main/scripts/generate_reference_data.py"
BUILTIN_SCRIPT = Path("./builtin/generate_reference_data.py")

BUILTIN_SCRIPT.parent.mkdir(parents=True, exist_ok=True)

if BUILTIN_SCRIPT.exists():
    print(f"ℹ️  Script already in built-in resources ({BUILTIN_SCRIPT.stat().st_size / 1024:.1f} KB) — skipping download.")
    print("   Re-delete the file and re-run this cell to force a fresh download.")
else:
    print("Downloading generate_reference_data.py into built-in resources...")
    urllib.request.urlretrieve(SCRIPT_URL, BUILTIN_SCRIPT)
    print(f"✅ Script saved to {BUILTIN_SCRIPT} ({BUILTIN_SCRIPT.stat().st_size / 1024:.1f} KB)")

In [ ]:
import re
from pathlib import Path

BUILTIN_SCRIPT = Path("./builtin/generate_reference_data.py")
OUTPUT_PATH    = Path("/lakehouse/default/Files/reference_data")

if not BUILTIN_SCRIPT.exists():
    raise FileNotFoundError(
        "generate_reference_data.py not found in built-in resources.\n"
        "Re-run the download cell above first."
    )

OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

# Redirect OUTPUT_DIR to the Lakehouse POSIX mount (works with any script version)
script_src  = BUILTIN_SCRIPT.read_text(encoding="utf-8")
patched_src = re.sub(
    r'OUTPUT_DIR\s*=\s*os\.environ\.get\([^)]+\)|OUTPUT_DIR\s*=\s*os\.path\.join\([^)]+\)',
    f'OUTPUT_DIR = r"{OUTPUT_PATH}"',
    script_src,
)

if f'OUTPUT_DIR = r"{OUTPUT_PATH}"' not in patched_src:
    raise RuntimeError(
        "OUTPUT_DIR patch failed — the script format may have changed.\n"
        "Delete ./builtin/generate_reference_data.py and re-download it."
    )

print(f"Generating reference data into Lakehouse: {OUTPUT_PATH}\n")
exec(patched_src, {"__file__": str(BUILTIN_SCRIPT), "__name__": "__main__"})

jsonl_files = sorted(OUTPUT_PATH.glob("*.jsonl"))
if not jsonl_files:
    raise RuntimeError("No JSONL files were written. Check the output above for errors.")

print(f"\n🎉 {len(jsonl_files)} files written successfully.")
print(f"   Lakehouse : {LAKEHOUSE_NAME}  ({LAKEHOUSE_ID})")
print(f"   Path      : Files/reference_data/")
print( "   Next      : Run Step 3 to create Delta tables from these files.")

---
## 📊 Step 3 — Create Delta Tables

Runs `01_load_reference_data.ipynb` as a child notebook, passing the Lakehouse name  
and data path as parameters. That notebook reads each JSONL file and creates a  
corresponding Delta table in the Lakehouse.

> This step may take 2–5 minutes to complete.

In [ ]:
notebookutils.notebook.run(
    "01_load_reference_data",
    timeout_seconds=600,
    arguments={
        "LAKEHOUSE_NAME":      LAKEHOUSE_NAME,
        "REFERENCE_DATA_PATH": "Files/reference_data",
    },
)
print("✅ Delta tables created successfully.")

---
<div align="center">

**Environment ready! 🚛💨**  
Open **`01_intro_to_ontologies.ipynb`** to begin the workshop.

</div>